# Task 27 — Solutions Walkthrough

**Topic:** 09 Data Analysis Process · **Level:** 🟠 Intermediate · **Type:** TASK SOLUTIONS

## What you are looking at

This is the worked reference for Task 27, the Data Gathering assessment — the notebook the course expects you to attempt first and compare against after. It solves all three problems the task set: **Problem 1** pulls three relational tables from a MySQL database into a three-sheet Excel workbook; **Problem 2** gathers cities from a public rapid API, paginating politely, and lands the records in a database table; **Problem 3** scrapes a smartphone listing page with BeautifulSoup across several pages and saves the result as JSON. Read it the right way — *after* your own attempt, problem by problem, comparing the shape of your staircase to theirs.

## The three solution shapes in one glance

**Relational → Excel.** `mysql.connector.connect(...)` opens the database, `pd.read_sql_query('SELECT * FROM invoices', conn)` turns each table into a DataFrame, and one `pd.ExcelWriter` writes three sheets in a single `with` block. **API → pandas → SQL.** A `requests` loop walks the API's own `links` navigation — not a hard-coded page counter — extending a shared `data` list, sleeping two seconds between calls, then `pd.DataFrame(data).to_sql("cities", con=engine)` writes it home. **Browser → CSV/JSON.** `find_all('div', class_="_2kHMtA")` targets the repeating product card, each field is extracted with `.find(...)` under a `try`, and the same extraction runs inside a `for` over `?page=1..6` with `time.sleep(2)` between pages.

## What to compare when you read it

Grade yourself on the seam-checks, not the imports: shapes printed after every read, `time.sleep` at every network seam, the `try` / `except` that keeps one missing field from killing the whole row, and the final `final_df.shape` proving all five pages combined. Where your pipeline matches, you are ready; where it differs, the difference is a lesson with your name on it.


In [1]:
import pandas as pd
import requests
from io import StringIO

### Problem-1:

You are given a SQL file link: https://drive.google.com/file/d/1WFt7B84LTHhMueoKmz8W-PRo7xXqmZf3/view?usp=share_link. Read the data by using the file and store it in a excel file. In this data, there are 3 tables named "invoices", "order_leads" and "sales_sql". So create 3 sheets to your excel file.

## Reading the Problem 1 solution

Three SQL tables, one workbook — the solution does it in exactly three moves and verifies each seam. **Connect**: `mysql.connector.connect(host=..., user=..., password=..., database=...)` opens the local MySQL instance that holds the task's tables. **Query**: one `pd.read_sql_query('SELECT * FROM invoices', conn)` per table turns `invoices`, `order_leads`, and `sales_sql` into DataFrames — pandas doing the SQL handshake, so no manual row-by-row copy is involved. **Write**: a single `with pd.ExcelWriter('supermarket.xlsx') as writer:` block writes each DataFrame with its own `sheet_name`, and the context manager closes the file for you.

The three lessons to steal. The `with` block is not decoration — it guarantees the workbook is flushed and closed even if a sheet errors. Each `read_sql_query` is a separate SELECT, keeping the pipeline readable. And the output filename is meaningful (`supermarket.xlsx`), so the deliverable is a file you can point at — which is the entire contract of the task. Compare your own attempt against these three moves and you will see exactly which seam you left unverified.


In [3]:
# code here
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(host="localhost",user="root",password="",database="temp_db")

invoices = pd.read_sql_query('SELECT * FROM invoices',conn)
order_lead = pd.read_sql_query('SELECT * FROM order_leads',conn)
sales_sql = pd.read_sql_query('SELECT * FROM sales_sql',conn)


with pd.ExcelWriter('supermarket.xlsx') as writer:
    invoices.to_excel(writer,sheet_name="invoices")
    order_lead.to_excel(writer,sheet_name="order_lead")
    sales_sql.to_excel(writer,sheet_name="sales_sql")

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 79.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for mysql.connector: filename=mysql_connector-2.2.9-cp38-cp38-linux_x86_64.whl size=247965 sha256=897a6a58f1ce532548feb2e5ac2d011c6f4249e74843229071ec847ac4cecb0d
  Stored in directory: /root/.cache/pip/wheels/57/e4/98/5feafb5c393dd2540e44b064a6f95832990d543e5b4f53ea8f
Successfully built mysql.connector


### Problem-2

Go to the site: https://rapidapi.com/wirefreethought/api/geodb-cities. From here, you have to grab the API and have to choose proper routes to get the cities of different countries. After getting the right API, hit that API and create a dataframe of all the cities that you can get by using the API. Then store the dataframe to a SQL. If you need to create an account or have to subscribe, then do that (it has free subscription but has some limitations. Use that free subscription and modify your accordingly to get all the data).  

## Reading the Problem 2 solution — setup

Problem 2's opening cells assemble the pieces before the loop begins, and every piece has a job. The `requests` call hits the **geo-db** admin-divisions endpoint with the two RapidAPI headers — `X-RapidAPI-Key` (your key) and `X-RapidAPI-Host` (the API's name) — and a GET with those headers is what authenticates the call.

Then the parsing splits the JSON response into three views of one document: `data` (the actual payload), `metadata` (where `totalCount` lives), and `links` (the API's own navigation — the `next` link that will drive the loop). Separating them now means the loop can read each without re-parsing the response.

A note on the plan: the free RapidAPI tier caps how much you can fetch; this solution works inside that limit rather than fighting it. Whatever page size the live API offers feeds directly into how many page-loops remain — which is why the next cells read the count and the links *before* looping, not inside it.


In [ ]:
### code here
### Rapid API 
import requests

url = "https://wft-geo-db.p.rapidapi.com/v1/geo/adminDivisions"

headers = {
    "X-RapidAPI-Key": "b0f75c828fmsh2b0480a62ab8d23p18178ajsn3267e033ae6a",
    "X-RapidAPI-Host": "wft-geo-db.p.rapidapi.com"
}

response = requests.request("GET", url, headers=headers)

data = response.json()['data']
links = response.json()['links']
metadata = response.json()['metadata']

In [ ]:
next_url = links[1]['href']

## The pagination loop — linking, not counting

The loop that follows is pagination done the robust way: it follows the API's own **`links` navigation** rather than trusting a guessed page timber. Each iteration rebuilds the URL from the recorded `next_url`, requests it, re-reads the response to pocket the *new* `next` link (`for item in links: if item['rel'] == 'next'`), extends the `data` list with that page's records, and sleeps two seconds.

Three details separate this from a naive `range` loop. **The stop condition is the API's** — the loop ends when the response offers no next link, so page counts changing upstream cannot break it. **Sleep before request**, not after the last: each call is a separate two-second gap, keeping the key alive under rate limits. **The accumulator is `data.extend(...)`** — one list appended page by page, no `concat` inside the loop, and the printout per iteration makes progress visible and failures audible.

The template to reuse anywhere: *read links, sleep, extend, repeat*. That is the polite, self-terminating shape of a production API gather, and exactly what the task's verification step is checking for.


In [ ]:
count = metadata['totalCount']//5

In [ ]:
import time


for i in range(1,count):
    
    num_iters += 1
    
    url = "https://wft-geo-db.p.rapidapi.com"
    page = url + next_url
    
    response = requests.request("GET", page, headers=headers)
    
    print(response.json())
    
    links = response.json()['links']
    
    for item in links:
        if item['rel'] == 'next':
            next_url = item['href']
            
    data.extend(response.json()['data'])
    
    time.sleep(2)
    

In [ ]:
cities = pd.DataFrame(data)

## Cities to SQL — the SQLAlchemy handshake

With `data` gathered, the solution converts the collected records into a DataFrame and loads it into MySQL with a library most candidates forget: **SQLAlchemy**. `from sqlalchemy import create_engine` builds a connection string in the standard form — `mysql+pymysql://root:@localhost/temp_db` — and `cities.to_sql("cities", con=engine, if_exists='append')` lets pandas speak to the database.

Two pieces of the recipe are worth memorising. The URL scheme `mysql+pymysql://` names the *dialect* (MySQL) and the *driver* (pymysql) in one string — SQLAlchemy's engine understands it directly, no separate empower. And `to_sql` is the mirror of `read_sql_query`: pandas writes the whole DataFrame in one call with `if_exists='append'` (add to the table) versus `'replace'` (rebuild it) — a choice that keeps re-runs safe.

This is the same engine idiom the database tables earlier used — and the reason the task's second deliverable, *data now in a queryable database*, works at all. If your own attempt used `mysql.connector` cursor loops, this is the upgrade: the DataFrame is the unit, not the row.


In [ ]:
import pymysql

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("mysql+pymysql://root:@localhost/temp_db")

cities.to_sql("cities",con=engine,if_exists='append')

### Problem 3:

Go to this url: https://www.flipkart.com/search?q=smartphones. This is the url to find phones in flipkart website. You have to extract the below things:
1. image url of the phone
2. name of the image
3. average ratings
4. total ratings
5. total reviews
6. discounted price
7. actual price

Extract all the phones which are available in this website. So you have to use the pagination concept. **Also after requesting every page through the url, please wait for a while (minimum 2-3 seconds), otherwise your IP address can be banned to access the flipkart website later.**

After collecting all the data, save that in a JSON file.

## Problem 3 — scraping the shopping page

Problem 3's brief is a shopping-list extraction: from Flipkart's smartphone search, pull each phone's image URL, name, rating, rating count, price, and (in the full task) actual price — across **multiple pages**, politely, and save the result as JSON.

The solution's architecture is the standard scraping triad. **Target the repeating block**: `soup.find_all('div', class_="_2kHMtA")` — one CSS class per product card — so every `phone` in the loop is one item. **Extract each field defensively**: `phone.find("img", class_="_396cs4")['src']` inside `try` / `except`, with `np.nan` on failure — a single missing element on one card must not crash the whole batch. **Assemble**: columns built as parallel lists, then stacked into one `DataFrame` named after the fields it actually holds.

The clue in the class names is the lesson: `_2kHMtA`, `_396cs4`, `_4rR01T` are Flipkart's rolling class strings. A scrape is only as durable as its selectors — and when a site's markup marches on, the script's *structure* (block, fields, fallbacks) is what keeps it repairable in minutes.


In [1]:
# code here
import requests
from bs4 import BeautifulSoup
import time

import pandas as pd
import numpy as np

In [3]:
webpage = requests.get('https://www.flipkart.com/search?q=smartphones&page=1').text

In [4]:
soup = BeautifulSoup(webpage,'lxml')

In [10]:
all_divs = soup.find_all('div', class_="_2kHMtA")

In [43]:
phone_image_url = []
phone_name = []
phone_rating = []
phone_num_ratings = []
phone_price = []

for phone in all_divs:
  try:
    phone_image_url.append(phone.find("img",class_="_396cs4")['src'])
  except:
    phone_image_url.append(np.nan)

  try:
    phone_name.append(phone.find("div",class_="_4rR01T").text.strip())
  except:
    phone_name.append(np.nan)

  try:
    phone_rating.append(phone.find("div",class_="_3LWZlK").text.strip())
  except:
    phone_rating.append(np.nan)

  try:
    phone_num_ratings.append(phone.find_all('span',class_='_2_R_DZ')[0].find_all('span')[0].find_all('span')[0].text.strip())
  except:
    phone_num_ratings.append(np.nan)

  try:
    phone_price.append(phone.find("div",class_="_30jeq3 _1_WHN1").text.strip())
  except:
    phone_price.append(np.nan)


df = pd.DataFrame({'phone_name':phone_name,'phone_price':phone_price,'phone_rating':phone_rating,'phone_num_rating':phone_num_ratings,'phone_img_url':phone_image_url})
df

                                     phone_name phone_price phone_rating  \
0   Infinix HOT 20 Play (Fantasy Purple, 64 GB)      ₹8,199          4.4   
1     Infinix HOT 20 Play (Racing Black, 64 GB)      ₹8,199          4.4   
2     Infinix HOT 20 Play (Aurora Green, 64 GB)      ₹8,199          4.4   
3                realme C33 (Sandy Gold, 32 GB)      ₹8,999          4.4   
4        Infinix HOT 20 Play (Luna Blue, 64 GB)      ₹8,199          4.4   
5                 realme C33 (Night Sea, 32 GB)      ₹8,999          4.4   
6         vivo T1 44W (Midnight Galaxy, 128 GB)     ₹14,499          4.5   
7              vivo T1 44W (Starry Sky, 128 GB)     ₹14,499          4.5   
8          Infinix Smart 6 HD (Aqua Sky, 32 GB)      ₹5,599          4.4   
9       Infinix Smart 6 HD (Origin Blue, 32 GB)      ₹5,599          4.4   
10      Infinix Smart 6 HD (Force Black, 32 GB)      ₹5,599          4.4   
11               vivo T1 44W (Ice Dawn, 128 GB)     ₹14,499          4.5   
12        vi

## The multi-page loop — same extraction, polite timing

The final cell generalises the single-page scrape into a **paged loop**: for `page` 1 through 5, it fetches `f'https://www.flipkart.com/search?q=smartphones&page={i}'`, re-parses with BeautifulSoup, re-extracts every product block into fresh lists, stacks them into a per-page DataFrame, and concatenates that page into `final_df` — then sleeps two seconds before the next request.

Three choices make the loop production-shaped. **The pagination is the URL** — `?page=1`, `?page=2`, ... — the simplest form of the pattern, and the loop *reads* the page number it needs rather than following links off-page. **The lists are rebuilt per page** — `phone_image_url = []` etc. are created inside the loop, then `clear()`ed, so no stale card from page three contaminates page four's frame. **Sleeping is non-negotiable**: `time.sleep(2)` after each page is not a nicety; the brief itself calls it out, because a banned IP ends the whole exercise.

And the closing proof is a contract with the data: `final_df.shape` prints `(120, 5)` — five pages worth of phones, all fields captured. Compare it with `(24, 5)` from the single page, and the loop has verified itself.


In [51]:
final_df = pd.DataFrame()

for i in range(1,6):

  webpage = requests.get('https://www.flipkart.com/search?q=smartphones&page={}'.format(i)).text
  soup = BeautifulSoup(webpage,'lxml')
  all_divs = soup.find_all('div', class_="_2kHMtA")

  phone_image_url = []
  phone_name = []
  phone_rating = []
  phone_num_ratings = []
  phone_price = []

  for phone in all_divs:
    try:
      phone_image_url.append(phone.find("img",class_="_396cs4")['src'])
    except:
      phone_image_url.append(np.nan)

    try:
      phone_name.append(phone.find("div",class_="_4rR01T").text.strip())
    except:
      phone_name.append(np.nan)

    try:
      phone_rating.append(phone.find("div",class_="_3LWZlK").text.strip())
    except:
      phone_rating.append(np.nan)

    try:
      phone_num_ratings.append(phone.find_all('span',class_='_2_R_DZ')[0].find_all('span')[0].find_all('span')[0].text.strip())
    except:
      phone_num_ratings.append(np.nan)

    try:
      phone_price.append(phone.find("div",class_="_30jeq3 _1_WHN1").text.strip())
    except:
      phone_price.append(np.nan)


  df = pd.DataFrame({'phone_name':phone_name,'phone_price':phone_price,'phone_rating':phone_rating,'phone_num_rating':phone_num_ratings,'phone_img_url':phone_image_url})
  final_df = final_df.append(df)

  phone_image_url.clear()
  phone_name.clear()
  phone_rating.clear()
  phone_num_ratings.clear()
  phone_price.clear()

  time.sleep(2)


In [52]:
final_df.shape

(120, 5)

In [46]:
df.shape

(24, 5)

In [54]:
final_df.isnull().sum()

phone_name          0
phone_price         0
phone_rating        2
phone_num_rating    2
phone_img_url       0
dtype: int64

## Summary

The solutions walked all three gathering shapes to completion: relational tables out of MySQL into a three-sheet Excel workbook; a paginated API pulled down via its own `links` navigation with two-second sleeps and a terminal `to_sql`; and a flipkart phone scrape run across five pages with defensive `try` blocks and a final `(120, 5)` shape to prove it. The seams were the story everywhere — shapes printed, sleep between calls, `np.nan` on missing fields, and a deliverable file at the end of every path.

Score your own Task 27 against this before moving on: did you verify counts, keep types honest, sleep, and make failures visible? Where the recipes line up, the assessment worked; where they diverge, the difference is the next thing to practise. The tools differ per problem — the discipline should not.
